# Session 07: Test the search and control who sees what

Score Kitara's three searches on its sixty test questions, find the stale and paraphrase failures, and run the permissions test with the users in users.csv. Then use Search Test Sheet.xlsx for your own thirty questions.

**Student version.** This is the team notebook. Leaders can follow it on the shared screen. Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("customers.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith(".csv")), "and the docs folder")

## Setup: build the search
This cell loads the Kitara documents, cuts them into chunks and builds one searcher. It takes about fifteen seconds because it reads the scans with OCR. Run it once.

In [ ]:
import pandas as pd
import kitara_search as ks

corpus = ks.load_corpus('.')
chunks = ks.build_chunks(corpus)
search = ks.Searcher(chunks)
tests = pd.read_csv('eval_questions.csv')
tests['kind'] = tests.note.str.split(';').str[0]
meta = corpus.set_index('doc_id')
print(len(tests), 'questions,', len(chunks), 'chunks')

## Task 1. Where does the right document rank?
Write `rank_of(method, question, gold)`. It runs the search for the top 10 chunks and returns the 1-based position of the first chunk from the gold document, or 0 when it is not in the top 10. Fill `tests['bm25']` with the rank for every question.

In [ ]:
def rank_of(method, question, gold):
    # Your turn: run search.search(question, method, k=10) and make a list of its .doc_id
    # Your turn: return the position of gold in that list (counting from 1), or 0 if it is missing
    pass

# Your turn: fill tests['bm25'] with rank_of for every question (zip tests.question and tests.gold_doc_id)


In [ ]:
assert tests.bm25.tolist().count(1) == 25, 'Keyword search should rank the right document first for 25 questions. Count positions from 1 and return 0 when it is missing'
assert ((tests.bm25 > 0) & (tests.bm25 <= 5)).sum() == 38, 'The right document should be in the top 5 for 38 questions'
print('Task 1 correct')

## Task 2. Score the three searches
Fill `tests['meaning']` and `tests['hybrid']` the same way. Then write `recall_at(col, k)` and `mrr_of(col)` and build a small table `scores` with recall at 3, recall at 5 and mean reciprocal rank for the three methods.

In [ ]:
# Your turn: fill tests['meaning'] and tests['hybrid'] with rank_of, as you did for bm25

def recall_at(col, k):
    # Your turn: the share of questions where the rank is above 0 and at most k
    pass

def mrr_of(col):
    # Your turn: the mean of 1 / rank, counting a rank of 0 as a score of 0
    pass

# Your turn: build scores, a table with rows recall@3, recall@5, MRR and a column for each method


In [ ]:
_missing = [v for v in ('scores',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert abs(scores.loc['recall@5', 'hybrid'] - 0.767) < 0.001, 'Hybrid recall at 5 should be 0.767'
assert abs(scores.loc['MRR', 'bm25'] - 0.514) < 0.001, 'Keyword mean reciprocal rank should be 0.514. A rank of 0 must count as 0, not 1 / 0'
assert scores.loc['MRR', 'hybrid'] < scores.loc['MRR', 'bm25'], 'Hybrid has the lower mean reciprocal rank on this test set'
print('Task 2 correct')

## Task 3. Stale versions
For the 8 stale questions, count how many times keyword search returns a superseded document first and store it in `n_old_first`. Then switch on `current_only=True` and count how many questions have the right document in the top 3, in `n_fixed`.

In [ ]:
stale = tests[tests.kind == 'stale']
status = meta.status   # document id -> current or superseded
# Your turn: n_old_first = how many stale questions get a superseded document as the first keyword result
# Your turn: n_fixed = how many have the gold document in the top 3 with current_only=True


In [ ]:
_missing = [v for v in ('n_fixed', 'n_old_first',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert n_old_first == 8, 'Keyword search should return a superseded document first for all 8 stale questions'
assert n_fixed == 8, 'With current_only=True the right document should be in the top 3 for all 8'
print('Task 3 correct')

## Task 4. The failure list
Hybrid search misses some questions in the top 10 (rank 0). Give each one a failure type: `Swahili`, `paraphrase` (the note column contains the word paraphrase) or `other`. Count the types in `failures`.

In [ ]:
tests['paraphrase'] = tests.note.str.contains('paraphrase')
missed = tests[tests.hybrid == 0]
# Your turn: write why(r) so it returns 'Swahili' when r.language is 'sw', else 'paraphrase' or 'other'
# Your turn: failures = the count of each type among the missed questions


In [ ]:
_missing = [v for v in ('failures', 'missed',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert len(missed) == 12, 'Hybrid search misses 12 questions in the top 10'
assert failures['paraphrase'] == 9 and failures['Swahili'] == 2 and failures['other'] == 1, 'Expected 9 paraphrase, 2 Swahili and 1 other'
print('Task 4 correct')

## Task 5. The permissions test
For each of the 6 restricted questions pick a user from `users.csv` whose groups are `all_staff` plus the asker group. Run hybrid search without and with `user_groups`. Count how many times the right document appears in the top 5 in each case: `unfiltered` and `filtered`.

In [ ]:
users = pd.read_csv('users.csv')
unfiltered = filtered = 0
for r in tests[tests.kind == 'restricted'].itertuples():
    want = 'all_staff' if r.asker_group == 'all_staff' else 'all_staff;' + r.asker_group
    # Your turn: take the groups of the first user in users whose groups equal want
    # Your turn: add 1 to unfiltered when the gold document is in the top 5 of a search with no user_groups
    # Your turn: add 1 to filtered when it is in the top 5 of a search with user_groups=groups
    pass
print(unfiltered, filtered)

In [ ]:
_missing = [v for v in ('filtered', 'unfiltered',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert unfiltered == 6, 'Without a filter the restricted document should appear in the top 5 for all 6 questions'
assert filtered == 0, 'With the filter no restricted document may appear. Pass user_groups=groups to search.search'
print('Task 5 correct')

## For the leader
Look at your results. The permission filter kept all six restricted documents out, and the stale and paraphrase failures remain. Who in your organisation would sign off this pilot for production, and which of these five results would you want to see before they do? Write your answer below.